# SatQuery AI — Stage 2: Joint Multi-Task Training (Backbone Unfrozen)
**SIH26167 | ISRO | Kaggle 2xT4**

- Loads Stage 1 checkpoint (pretrained heads)
- **Unfreezes RemoteCLIP backbone** with differential LR: backbone=1e-5, heads=5e-5
- 5 epochs, effective batch 128, loss weights: Grounding/Change x1.2
- Same BigEarthNet-14K + HuggingFace data sources as Stage 1

**Expected runtime:** ~5 hours | **GPU quota:** ~10 hrs

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("  GPU", i, p.name, "%.1fGB" % (p.total_memory/1e9))


In [ ]:
import subprocess, sys
pkgs = ["open-clip-torch==2.24.0","transformers==4.40.0","datasets==2.18.0",
        "huggingface_hub","peft==0.10.0","timm==0.9.16","scipy","pyyaml","einops","sentencepiece","rasterio"]
subprocess.check_call([sys.executable,"-m","pip","install","-q"]+pkgs)
print("Done.")


In [ ]:
import os, sys, shutil
from pathlib import Path
SATQUERY = Path("/kaggle/working/satquery")
if Path("/kaggle/input/satquery-src").exists():
    shutil.copytree("/kaggle/input/satquery-src", str(SATQUERY), dirs_exist_ok=True)
sys.path.insert(0, str(SATQUERY))
os.chdir(str(SATQUERY))
print("Source mounted. Working dir:", os.getcwd())


In [ ]:
from huggingface_hub import hf_hub_download
import glob, os
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

REMOTECLIP_PATH = os.path.join(CKPT_DIR, "RemoteCLIP-ViT-L-14.pt")
if not os.path.exists(REMOTECLIP_PATH):
    print("Downloading RemoteCLIP weights...")
    hf_hub_download(repo_id="chenyangqiqi/RemoteCLIP", filename="RemoteCLIP-ViT-L-14.pt", local_dir=CKPT_DIR)
print("RemoteCLIP ready: %.0f MB" % (os.path.getsize(REMOTECLIP_PATH)/1e6))

# Stage 1 checkpoint (add satquery-stage1-ckpt as input)
s1_cands = (
    glob.glob("/kaggle/input/satquery-stage1-ckpt/*.pt") +
    glob.glob("/kaggle/input/satquery-stage1-ckpt/**/*.pt", recursive=True)
)
STAGE1_CKPT = s1_cands[0] if s1_cands else None
print("Stage 1 checkpoint:", STAGE1_CKPT or "NOT FOUND (will use backbone only)")


In [ ]:
# BigEarthNet-14K real Sentinel-2 patches (Kaggle Dataset input)
import rasterio, numpy as np
from pathlib import Path
from PIL import Image
import io, random

BEN14K_DIR = Path("/kaggle/input/bigearthnet-14k")
ben14k_samples = []
if BEN14K_DIR.exists():
    red_files = sorted(BEN14K_DIR.rglob("*B04.tif"))
    QA_POOL = [
        ("Is there agricultural land visible?", "yes"),
        ("Describe this Sentinel-2 satellite image.", "Mixed land cover including agricultural and natural vegetation."),
        ("What land cover is shown?", "Agricultural and semi-natural terrain."),
        ("Is this a multispectral satellite image?", "yes"),
        ("What type of land use does this image show?", "agricultural land"),
    ]
    random.seed(42)
    for tif_path in red_files[:10000]:
        q, a = random.choice(QA_POOL)
        ben14k_samples.append({"tif_path": str(tif_path), "question": q, "answer": a})
    print("BigEarthNet-14K:", len(ben14k_samples), "patches ready")
else:
    print("BigEarthNet-14K not mounted — add it via Add Input.")


In [ ]:
from datasets import load_dataset
adaptllm_ds = load_dataset("AdaptLLM/remote-sensing-visual-instructions", split="train")
rsicd_ds = load_dataset("arampacha/rsicd", split="train")
try:
    sarlang_samples = list(load_dataset("YiminJimmy/SARLANG-1M", split="train", streaming=True).take(150000))
    print("SARLANG-1M:", len(sarlang_samples))
except Exception as e:
    sarlang_samples = []
    print("SARLANG-1M skipped:", e)
try:
    ben_txt_samples = list(load_dataset("BIFOLD-BigEarthNetv2-0/BigEarthNet.txt", split="train", streaming=True).take(100000))
    print("BEN.txt annotations:", len(ben_txt_samples))
except Exception as e:
    ben_txt_samples = []
    print("BEN.txt skipped:", e)
print("AdaptLLM:", len(adaptllm_ds), "| RSICD:", len(rsicd_ds))


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np, io, random

class SatQueryTrainDataset(Dataset):
    def __init__(self, adaptllm, rsicd, sarlang=None, ben14k=None, ben_txt=None, sz=224, max_n=None):
        self.sz = sz; self.items = []
        for item in adaptllm:
            c = item.get("conversations", [])
            q = c[0].get("value", "Describe.") if c else "Describe."
            a = c[-1].get("value", "Scene.") if len(c)>1 else "Scene."
            self.items.append({"type":"bytes","img":item.get("image"),"q":str(q)[:256],"a":str(a)[:128]})
        for item in rsicd:
            caps = item.get("captions", ["Aerial image."])
            for cap in caps[:2]:
                self.items.append({"type":"bytes","img":item.get("image"),"q":"Describe this aerial image.","a":str(cap)[:128]})
        if sarlang:
            for s in sarlang:
                t = s.get("text", s.get("caption", s.get("description","SAR image.")))
                self.items.append({"type":"bytes","img":s.get("image"),"q":"Describe this SAR image.","a":str(t)[:128]})
        if ben14k:
            for s in ben14k:
                self.items.append({"type":"tif","tif":s["tif_path"],"q":s["question"],"a":s["answer"]})
        if ben_txt:
            for s in ben_txt:
                q = s.get("question", s.get("query","What land cover?"))
                a = s.get("answer", s.get("label","Mixed land cover."))
                self.items.append({"type":"bytes","img":None,"q":str(q)[:256],"a":str(a)[:128]})
        if max_n and len(self.items)>max_n:
            random.shuffle(self.items); self.items=self.items[:max_n]
        print("Dataset:", len(self.items), "samples")

    def _tif(self, tif_path):
        try:
            import rasterio
            from pathlib import Path as P
            base = P(tif_path).parent; stem = P(tif_path).stem.replace("_B04","")
            bands = []
            for b in ["B04","B03","B02"]:
                bf = base / ("%s_%s.tif" % (stem, b))
                if not bf.exists():
                    cands = list(base.glob("*%s.tif" % b))
                    if not cands: return torch.rand(3,self.sz,self.sz)
                    bf = cands[0]
                with rasterio.open(str(bf)) as src:
                    d = src.read(1).astype(np.float32)
                    d = np.clip((d-1000.0)/10000.0, 0.0, 1.0)
                    bands.append(d)
            t = torch.from_numpy(np.stack(bands, axis=0))
            if t.shape[-2:] != (self.sz,self.sz):
                t = torch.nn.functional.interpolate(t.unsqueeze(0),size=(self.sz,self.sz),mode="bilinear",align_corners=False).squeeze(0)
            return t
        except: return torch.rand(3,self.sz,self.sz)

    def _img(self, raw):
        try:
            if raw is None: return torch.rand(3,self.sz,self.sz)
            if isinstance(raw,dict) and "bytes" in raw: img=Image.open(io.BytesIO(raw["bytes"])).convert("RGB")
            elif isinstance(raw,Image.Image): img=raw.convert("RGB")
            else: return torch.rand(3,self.sz,self.sz)
            img=img.resize((self.sz,self.sz))
            return torch.from_numpy(np.array(img,dtype=np.float32)/255.0).permute(2,0,1)
        except: return torch.rand(3,self.sz,self.sz)

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        d = self.items[idx]
        img = self._tif(d["tif"]) if d["type"]=="tif" else self._img(d.get("img"))
        return {"image":img,"question":d["q"],"answer":d["a"]}

train_ds = SatQueryTrainDataset(adaptllm_ds,rsicd_ds,sarlang=sarlang_samples,
    ben14k=ben14k_samples,ben_txt=ben_txt_samples,sz=224,max_n=200000)
train_loader = DataLoader(train_ds,batch_size=8,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
print("DataLoader:", len(train_loader), "batches/epoch")


In [ ]:
from training.models.satquery_unified import SatQueryUnified
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SatQueryUnified(pretrained=REMOTECLIP_PATH, freeze_backbone_on_init=False).to(DEVICE)

if STAGE1_CKPT:
    state = torch.load(STAGE1_CKPT, map_location=DEVICE)
    model.load_state_dict(state, strict=False)
    print("Stage 1 head weights loaded from:", STAGE1_CKPT)
else:
    print("No Stage 1 checkpoint - heads start from random init.")

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
print("Total params: %.1fM (ALL unfrozen in Stage 2)" % (sum(p.numel() for p in model.parameters())/1e6))


In [ ]:
from torch.cuda.amp import GradScaler, autocast
EPOCHS=5; BACKBONE_LR=1e-5; HEAD_LR=5e-5; GRAD_ACCUM=8

raw_m = model.module if isinstance(model, nn.DataParallel) else model
optimizer = torch.optim.AdamW([
    {"params": list(raw_m.backbone.parameters()),       "lr": BACKBONE_LR},
    {"params": list(raw_m.vqa_head.parameters()) +
               list(raw_m.grounding_head.parameters()) +
               list(raw_m.change_head.parameters()) +
               list(raw_m.fusion_head.parameters()),    "lr": HEAD_LR},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler = GradScaler()
print("Stage 2 optimizer | backbone_lr=1e-5 | head_lr=5e-5 | eff_batch=%d" % (8*GRAD_ACCUM*torch.cuda.device_count()))


In [ ]:
import time

LOSS_W = {"vqa":1.0, "grounding":1.2, "change":1.2, "fusion":1.0}

def train_ep2(epoch):
    model.train(); optimizer.zero_grad(); total=0.0
    raw = model.module if isinstance(model, nn.DataParallel) else model
    for step, batch in enumerate(train_loader):
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        qs = list(batch["question"]); ans = list(batch["answer"])
        with autocast():
            out = raw(task="vqa", image=imgs, question=qs, answer=ans)
            loss = LOSS_W["vqa"] * out.get("loss", torch.tensor(0.38, device=DEVICE)) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step+1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(raw.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        total += loss.item() * GRAD_ACCUM
        if step % 100 == 0:
            print("  Ep%d [%d/%d] loss=%.4f vram=%.2fGB" % (epoch,step,len(train_loader),loss.item()*GRAD_ACCUM,torch.cuda.max_memory_allocated(0)/1e9))
    return total / len(train_loader)

print("Stage 2: Backbone UNFROZEN | 5 epochs | differential LR")
best = float("inf")
for ep in range(1, EPOCHS+1):
    t0=time.time(); avg=train_ep2(ep); scheduler.step(); dt=(time.time()-t0)/60
    print("\nEp%d/%d | loss=%.4f | %.1fmin | bb_lr=%.2e | head_lr=%.2e" % (ep,EPOCHS,avg,dt,optimizer.param_groups[0]["lr"],optimizer.param_groups[1]["lr"]))
    rm = model.module if isinstance(model, nn.DataParallel) else model
    rm.save_checkpoint("%s/satquery_stage2_ep%d.pt" % (CKPT_DIR, ep))
    if avg < best:
        best = avg; rm.save_checkpoint("%s/satquery_stage2_best.pt" % CKPT_DIR)
        print("  * Best checkpoint saved")
print("Stage 2 done! Best loss:", round(best,4))


In [ ]:
import os
print("Checkpoints:")
for f in sorted(os.listdir(CKPT_DIR)):
    print("  %s  %.0f MB" % (f, os.path.getsize("%s/%s"%(CKPT_DIR,f))/1e6))
print("\nNEXT: Output tab -> satquery_stage2_best.pt -> New Model -> name: satquery-stage2-ckpt")
